# Merge single cells from CellProfiler outputs using CytoTable

In [1]:
import argparse
import os
import pathlib
import pprint
import sys
import uuid

import natsort
import pandas as pd
from cytotable import convert, presets
from timelapse_utils.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)
from timelapse_utils.profiling_utils.sc_extraction_utils import add_single_cell_count_df

root_dir, in_notebook = init_notebook()
if in_notebook:
    import tqdm.notebook as tqdm
else:
    import tqdm

from parsl.config import Config
from parsl.executors import HighThroughputExecutor

## Set paths and variables

All paths must be string but we use pathlib to show which variables are paths

In [2]:
image_base_dir = bandicoot_check(
    bandicoot_mount_path=pathlib.Path(f"{os.path.expanduser('~')}/mnt/bandicoot/"),
    root_dir=root_dir,
)
image_base_dir = pathlib.Path(
    f"{image_base_dir}/live_cell_timelapse_pyroptosis_project_data/processed_data/"
).resolve(strict=True)
extracted_features_dir = pathlib.Path(
    f"{image_base_dir}/3.extracted_features/"
).resolve(strict=True)

In [3]:
# type of file output from CytoTable (currently only parquet)
dest_datatype = "parquet"

# directory where parquet files are saved to
output_dir = pathlib.Path(f"{image_base_dir}/4.converted_profiles/").resolve()
output_dir.mkdir(exist_ok=True, parents=True)

In [4]:
# well_fov_timepoints
well_fov_timepoints = [
    x for x in tqdm.tqdm(extracted_features_dir.glob("*")) if x.is_dir()
]
well_fov_timepoints = natsort.natsorted(well_fov_timepoints)[:10]
well_fov_timepoints_sqlites = [
    list(x.glob("**/*.sqlite"))[0]
    for x in tqdm.tqdm(well_fov_timepoints)
    if len(list(x.glob("**/*.sqlite"))) > 0
]
well_fov_timepoints_sqlites = natsort.natsorted(well_fov_timepoints_sqlites)

0it [00:00, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

In [5]:
sqlite_file_path = well_fov_timepoints_sqlites[0]
sqlite_file_path

PosixPath('/home/lippincm/mnt/bandicoot/live_cell_timelapse_pyroptosis_project_data/processed_data/3.extracted_features/B2_1_T0001/B2_1_T0001.sqlite')

In [6]:
dest_path = output_dir / str(sqlite_file_path.stem)
dest_path.mkdir(exist_ok=True, parents=True)
dest_path = dest_path / f"{sqlite_file_path.stem}.{dest_datatype}"
print(f"Destination path: {dest_path}")

Destination path: /home/lippincm/mnt/bandicoot/live_cell_timelapse_pyroptosis_project_data/processed_data/4.converted_profiles/B2_1_T0001/B2_1_T0001.parquet


## set config joins for each preset

In [7]:
# preset configurations based on typical CellProfiler outputs
preset = "cellprofiler_sqlite_pycytominer"
presets.config[preset][
    "CONFIG_JOINS"
    # remove Image_Metadata_Plate from SELECT as this metadata was not extracted from file names
    # add Image_Metadata_FOV as this is an important metadata when finding where single cells are located
] = """WITH Per_Image_Filtered AS (
                SELECT
                    Metadata_ImageNumber,
                    Image_URL_CL488,
                    Image_URL_CL640,
                    Image_URL_NucleoLive,
                    Image_URL_BF,
                    Image_URL_SYTOXGreen,
                FROM
                    read_parquet('per_image.parquet')
                )
            SELECT
                *
            FROM
                Per_Image_Filtered AS per_image
            LEFT JOIN read_parquet('per_cytoplasm.parquet') AS per_cytoplasm ON
                per_cytoplasm.Metadata_ImageNumber = per_image.Metadata_ImageNumber
            LEFT JOIN read_parquet('per_cells.parquet') AS per_cells ON
                per_cells.Metadata_ImageNumber = per_cytoplasm.Metadata_ImageNumber
                AND per_cells.Metadata_Cells_Number_Object_Number = per_cytoplasm.Metadata_Cytoplasm_Parent_Cells
            LEFT JOIN read_parquet('per_nuclei.parquet') AS per_nuclei ON
                per_nuclei.Metadata_ImageNumber = per_cytoplasm.Metadata_ImageNumber
                AND per_nuclei.Metadata_Nuclei_Number_Object_Number = per_cytoplasm.Metadata_Cytoplasm_Parent_Nuclei
            """

## Convert SQLite file and merge single cell objects into parquet file

This was not run to completion as we use the nbconverted python file for full run.

In [9]:
if dest_path.exists():
    dest_path.unlink()
# merge single cells and output as parquet file
convert(
    source_path=sqlite_file_path,
    dest_path=dest_path,
    dest_datatype=dest_datatype,
    preset=preset,
    parsl_config=Config(
        executors=[HighThroughputExecutor()],
        run_dir=f"cytotable_runinfo/{uuid.uuid4().hex}",
    ),
    chunk_size=1000,
)

PosixPath('/home/lippincm/mnt/bandicoot/live_cell_timelapse_pyroptosis_project_data/processed_data/4.converted_profiles/B2_1_T0001/B2_1_T0001.parquet')

In [ ]:
df = pd.read_parquet(dest_path)
# set columns to drop
df.drop(
    columns=[
        "Metadata_ImageNumber_1",
        "Metadata_ImageNumber_2",
        "Metadata_ImageNumber_3",
    ],
    inplace=True,
)
df.columns[:25]

Index(['Metadata_ImageNumber', 'Metadata_Cells_Number_Object_Number',
       'Metadata_Cytoplasm_Parent_Cells', 'Metadata_Cytoplasm_Parent_Nuclei',
       'Metadata_Nuclei_Number_Object_Number', 'Image_URL_BF',
       'Image_URL_CL488', 'Image_URL_CL640', 'Image_URL_NucleoLive',
       'Image_URL_SYTOXGreen', 'Cytoplasm_AreaShape_Area',
       'Cytoplasm_AreaShape_BoundingBoxArea',
       'Cytoplasm_AreaShape_BoundingBoxMaximum_X',
       'Cytoplasm_AreaShape_BoundingBoxMaximum_Y',
       'Cytoplasm_AreaShape_BoundingBoxMinimum_X',
       'Cytoplasm_AreaShape_BoundingBoxMinimum_Y',
       'Cytoplasm_AreaShape_Center_X', 'Cytoplasm_AreaShape_Center_Y',
       'Cytoplasm_AreaShape_CentralMoment_0_0',
       'Cytoplasm_AreaShape_CentralMoment_0_1',
       'Cytoplasm_AreaShape_CentralMoment_0_2',
       'Cytoplasm_AreaShape_CentralMoment_0_3',
       'Cytoplasm_AreaShape_CentralMoment_1_0',
       'Cytoplasm_AreaShape_CentralMoment_1_1',
       'Cytoplasm_AreaShape_CentralMoment_1_2'],
   

In [ ]:
Nuclei_Children_Cells_Count
Nuclei_Children_Cytoplasm_Count
Nuclei_AreaShape_BoundingBoxArea

In [ ]:
[
    x
    for x in df.columns
    if "center_" in x.lower() or "location" in x.lower() or "boundingbox" in x.lower()
]

['Cytoplasm_AreaShape_BoundingBoxArea',
 'Cytoplasm_AreaShape_BoundingBoxMaximum_X',
 'Cytoplasm_AreaShape_BoundingBoxMaximum_Y',
 'Cytoplasm_AreaShape_BoundingBoxMinimum_X',
 'Cytoplasm_AreaShape_BoundingBoxMinimum_Y',
 'Cytoplasm_AreaShape_Center_X',
 'Cytoplasm_AreaShape_Center_Y',
 'Cytoplasm_Location_CenterMassIntensity_X_BF',
 'Cytoplasm_Location_CenterMassIntensity_X_CL488',
 'Cytoplasm_Location_CenterMassIntensity_X_CL640',
 'Cytoplasm_Location_CenterMassIntensity_X_NucleoLive',
 'Cytoplasm_Location_CenterMassIntensity_X_SYTOXGreen',
 'Cytoplasm_Location_CenterMassIntensity_Y_BF',
 'Cytoplasm_Location_CenterMassIntensity_Y_CL488',
 'Cytoplasm_Location_CenterMassIntensity_Y_CL640',
 'Cytoplasm_Location_CenterMassIntensity_Y_NucleoLive',
 'Cytoplasm_Location_CenterMassIntensity_Y_SYTOXGreen',
 'Cytoplasm_Location_CenterMassIntensity_Z_BF',
 'Cytoplasm_Location_CenterMassIntensity_Z_CL488',
 'Cytoplasm_Location_CenterMassIntensity_Z_CL640',
 'Cytoplasm_Location_CenterMassIntensity

In [ ]:
location_metadata = []

In [ ]:
# set columns to move to Metadata_ prefix
file_metadata = [
    "Image_URL_BF",
    "Image_URL_CL488",
    "Image_URL_CL640",
    "Image_URL_NucleoLive",
    "Image_URL_SYTOXGreen",
]

In [ ]:
print(f"Merged and converted {pathlib.Path(dest_path).name}!")
print(f"Saved to {dest_path}")
df = pd.read_parquet(dest_path)
df["Metadata_Well_Time"] = df["Image_Metadata_Well"] + "_" + df["Image_Metadata_Time"]
print(f"Shape of {pathlib.Path(dest_path).name}: {df.shape}")
df.head()

Merged and converted B2_3_T0036.parquet!
Saved to /home/lippincm/mnt/bandicoot/live_cell_timelapse_pyroptosis_project_data/processed_data/4.converted_profiles/B2_3_T0036/B2_3_T0036.parquet


KeyError: 'Image_Metadata_Well'

In [ ]:
Metadata_number_of_singlecells_df = (
    df.groupby("Metadata_Well_Time")
    .value_counts()
    .reset_index(name="Metadata_number_of_singlecells")
)
# merge the number of single cells with the original dataframe
df = df.merge(
    Metadata_number_of_singlecells_df, on=["Metadata_Well_Time", "Metadata_Well_Time"]
)
df.head()

,Metadata_ImageNumber,Image_Metadata_FOV,Image_Metadata_Time,Image_Metadata_Well,Metadata_Cells_Number_Object_Number,Metadata_Cytoplasm_Parent_Cells,Metadata_Cytoplasm_Parent_Nuclei,Metadata_ImageNumber_1,Metadata_ImageNumber_2,Metadata_ImageNumber_3,...,Nuclei_Texture_Variance_DNA_3_00_256,Nuclei_Texture_Variance_DNA_3_01_256,Nuclei_Texture_Variance_DNA_3_02_256,Nuclei_Texture_Variance_DNA_3_03_256,Nuclei_Texture_Variance_GSDM_3_00_256,Nuclei_Texture_Variance_GSDM_3_01_256,Nuclei_Texture_Variance_GSDM_3_02_256,Nuclei_Texture_Variance_GSDM_3_03_256,Metadata_Well_Time,Metadata_number_of_singlecells
0,1,0001,00,0052,1.0,1,1,1,1.0,1.0,...,0.184131,0.188453,0.177188,0.181128,0.103136,0.115436,0.113431,0.134241,0052_00,502
1,1,0001,00,0052,2.0,2,17,1,1.0,1.0,...,0.212552,0.217013,0.210907,0.213728,0.410092,0.382479,0.408034,0.422193,0052_00,502
2,1,0001,00,0052,3.0,3,2,1,1.0,1.0,...,0.232724,0.223048,0.226669,0.223174,0.090572,0.087780,0.096994,0.087674,0052_00,502
3,1,0001,00,0052,4.0,4,3,1,1.0,1.0,...,0.437401,0.436013,0.421019,0.423367,0.000000,0.000000,0.000000,0.000000,0052_00,502
4,1,0001,00,0052,5.0,5,32,1,1.0,1.0,...,0.473553,0.457916,0.465556,0.472717,0.000000,0.000000,0.000000,0.000000,0052_00,502


In [ ]:
# cast to int
df["Metadata_number_of_singlecells"] = df["Metadata_number_of_singlecells"].astype(int)
df.to_parquet(dest_path)

print(f"Shape of {pathlib.Path(dest_path).name}: {df.shape}")
print(f"Added single cell count as metadata to {pathlib.Path(dest_path).name}!")

Shape of pyroptosis_timelapse.parquet: (12384, 2780)
Added single cell count as metadata to pyroptosis_timelapse.parquet!


In [ ]:
df.head()

,Metadata_ImageNumber,Image_Metadata_FOV,Image_Metadata_Time,Image_Metadata_Well,Metadata_Cells_Number_Object_Number,Metadata_Cytoplasm_Parent_Cells,Metadata_Cytoplasm_Parent_Nuclei,Metadata_ImageNumber_1,Metadata_ImageNumber_2,Metadata_ImageNumber_3,...,Nuclei_Texture_Variance_DNA_3_00_256,Nuclei_Texture_Variance_DNA_3_01_256,Nuclei_Texture_Variance_DNA_3_02_256,Nuclei_Texture_Variance_DNA_3_03_256,Nuclei_Texture_Variance_GSDM_3_00_256,Nuclei_Texture_Variance_GSDM_3_01_256,Nuclei_Texture_Variance_GSDM_3_02_256,Nuclei_Texture_Variance_GSDM_3_03_256,Metadata_Well_Time,Metadata_number_of_singlecells
0,1,0001,00,0052,1.0,1,1,1,1.0,1.0,...,0.184131,0.188453,0.177188,0.181128,0.103136,0.115436,0.113431,0.134241,0052_00,502
1,1,0001,00,0052,2.0,2,17,1,1.0,1.0,...,0.212552,0.217013,0.210907,0.213728,0.410092,0.382479,0.408034,0.422193,0052_00,502
2,1,0001,00,0052,3.0,3,2,1,1.0,1.0,...,0.232724,0.223048,0.226669,0.223174,0.090572,0.087780,0.096994,0.087674,0052_00,502
3,1,0001,00,0052,4.0,4,3,1,1.0,1.0,...,0.437401,0.436013,0.421019,0.423367,0.000000,0.000000,0.000000,0.000000,0052_00,502
4,1,0001,00,0052,5.0,5,32,1,1.0,1.0,...,0.473553,0.457916,0.465556,0.472717,0.000000,0.000000,0.000000,0.000000,0052_00,502
